# Hálózatkészítés PCA-alapú hasonlósággal (Rich Months)

Ez a notebook a havi klímaadatokból készít hálózatot. A hónapok (csomópontok) közötti hasonlóságot egy PCA (főkomponens-analízis) alapú módszerrel határozzuk meg.

A folyamat lépései:
1. **Adatbetöltés**: Betöltjük a havi klímaadatokat (átlaghőmérsékletek, stb.).
2. **Adatmátrix előkészítése**: A havi adatokból egy mátrixot készítünk, ahol a sorok a hónapok, az oszlopok pedig a klimatikus változók.
3. **Standardizálás**: A változókat standardizáljuk (z-score transzformáció), hogy mindegyiknek 0 legyen az átlaga és 1 a szórása. Ez fontos a PCA számára.
4. **PCA**: Elvégezzük a főkomponens-analízist a standardizált adatokon.
5. **Score-ok kinyerése**: A PCA eredményéből kinyerjük az egyes hónapokhoz tartozó score-okat (koordinátákat a főkomponens-térben).
6. **Hasonlósági mátrix számítása**: A hónapok PC score-jai közötti euklideszi távolságot számoljuk. Ezt a távolságot egy [0, 1] intervallumra skálázzuk, majd hasonlósággá alakítjuk (1 - skálázott távolság).
7. **Hálózatépítés**: A számított hasonlósági mátrix alapján létrehozzuk a hálózatot, ahol egy küszöbértéknél nagyobb hasonlóság esetén él jön létre a hónapok között.

In [1]:
import sys
sys.path.append('../..')

import pandas as pd
import numpy as np
import networkx as nx
import itertools
import json
import os

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform

from networks.utils.get_rich_monthly_nodes import get_rich_monthly_nodes
from networks.utils.prune_to_average_degree import prune_to_average_degree

## 1. Hasonlósági Mátrix Számítása (PCA)

In [2]:
def calculate_pca_similarity_matrix(monthly_nodes, n_components=2):
    """
    Kiszámítja a hasonlósági mátrixot és a PCA komponens súlyokat a havi adatokból.
    
    Argumentumok:
    monthly_nodes (dict): Kulcsok a hónapok ('YYYY-MM'), értékek a havi klímaadatok.
    n_components (int): A PCA-hoz használt főkomponensek száma.
    
    Visszatérési érték:
    tuple: (similarity_matrix, months, component_weights)
    """
    months = list(monthly_nodes.keys())
    sample_month_data = monthly_nodes[months[0]]
    feature_keys = [key for key, value in sample_month_data.items() if isinstance(value, (int, float))]
    
    print(f'Felhasznált változók a PCA-hoz: {feature_keys}')
    
    X = np.array([[monthly_nodes[month][key] for key in feature_keys] for month in months])
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(X_scaled)
    
    print(f'Magyarázott variancia aránya a komponensekkel: {pca.explained_variance_ratio_}')
    
    # Komponens súlyok kinyerése
    component_weights = {}
    for i, component in enumerate(pca.components_):
        component_weights[f'component_{i+1}'] = dict(zip(feature_keys, component))
    
    distances = pdist(scores, metric='euclidean')
    
    if distances.max() - distances.min() > 0:
        scaled_distances = (distances - distances.min()) / (distances.max() - distances.min())
    else:
        scaled_distances = np.zeros_like(distances)
    
    similarity_matrix = 1 - squareform(scaled_distances)
    
    return similarity_matrix, months, component_weights

## 2. Globális Hálózatok Létrehozása

In [3]:
def create_all_global_networks_pca(specifications, target_avg_degree_factor=3):
    threshold_json_filename = 'network_rich_thresholds.json'
    weights_json_filename = 'pca_component_weights.json'
    
    if os.path.exists(threshold_json_filename):
        with open(threshold_json_filename, 'r') as f: threshold_store = json.load(f)
    else: threshold_store = {}
        
    if os.path.exists(weights_json_filename):
        with open(weights_json_filename, 'r') as f: weights_store = json.load(f)
    else: weights_store = {}

    for city, start, end in specifications:
        print(f'--- Munka kezdete: {city} : {start} - {end} ---')
        
        monthly_nodes = get_rich_monthly_nodes(city, start, end)
        
        similarity_matrix, months, weights = calculate_pca_similarity_matrix(monthly_nodes)
        
        json_key = f'{city}_{start}_{end}'
        weights_store[json_key] = weights
        
        G = nx.Graph()
        for month, data in monthly_nodes.items():
            G.add_node(month, **data)

        for i, j in itertools.combinations(range(len(months)), 2):
            score = similarity_matrix[i, j]
            if score > 0:
                G.add_edge(months[i], months[j], weight=score)

        print(f'Kezdeti hálózat - Csomópontok: {G.number_of_nodes()}, Élek: {G.number_of_edges()}')

        num_years = int(end[:4]) - int(start[:4]) + 1
        target_avg_degree = num_years * target_avg_degree_factor - 1
        
        pruned_graph = prune_to_average_degree(G, target_avg_degree=target_avg_degree)

        if pruned_graph.number_of_edges() > 0:
            edge_weights = [d['weight'] for _, _, d in pruned_graph.edges(data=True)]
            final_threshold = float(min(edge_weights))
        else:
            final_threshold = 1.0
        
        threshold_store[json_key] = final_threshold
        with open(threshold_json_filename, 'w') as f: json.dump(threshold_store, f, indent=4)
        print(f'Küszöbérték ({final_threshold:.4f}) mentve: {json_key}')
        
        with open(weights_json_filename, 'w') as f: json.dump(weights_store, f, indent=4)
        print(f'PCA súlyok mentve: {json_key}')
        
        for node, data in pruned_graph.nodes(data=True):
            try:
                year, month = str(node).split('-')
                data['year'] = int(year)
                data['month'] = int(month)
            except ValueError:
                data['year'] = None
                data['month'] = None
        
        output_dir = '../global_networks/rich_global/'
        os.makedirs(output_dir, exist_ok=True)
        pruned_path = os.path.join(output_dir, f'{city}_{start}_{end}.graphml')
        nx.write_graphml(pruned_graph, pruned_path)
        print(f'Hálózat mentve: {pruned_path}')
        print('-'*40)

## 3. Futtatás

In [4]:
specifications = [
    ('Cluj', '1961-01', '1990-12'),
    ('Cluj', '1995-01', '2024-12'),
    ('Cluj', '1961-01', '2024-12'),
    ('Bacskatopolya', '1961-01', '1990-12'),
    ('Bacskatopolya', '1995-01', '2024-12'),
    ('Bacskatopolya', '1961-01', '2024-12'),
    ('Brasov', '1961-01', '1990-12'),
    ('Brasov', '1995-01', '2024-12'),
    ('Brasov', '1961-01', '2024-12'),
    ('Deva', '1961-01', '1990-12'),
    ('Deva', '1995-01', '2024-12'),
    ('Deva', '1961-01', '2024-12'),
    ('Gheorgheni', '1961-01', '1990-12'),
    ('Gheorgheni', '1995-01', '2024-12'),
    ('Gheorgheni', '1961-01', '2024-12'),
    ('Gyor', '1961-01', '1990-12'),
    ('Gyor', '1995-01', '2024-12'),
    ('Gyor', '1961-01', '2024-12'),
    ('Kassa', '1961-01', '1990-12'),
    ('Kassa', '1995-01', '2024-12'),
    ('Kassa', '1961-01', '2024-12'),
    ('Kecskemet', '1961-01', '1990-12'),
    ('Kecskemet', '1995-01', '2024-12'),
    ('Kecskemet', '1961-01', '2024-12'),
    ('Keszthely', '1961-01', '1990-12'),
    ('Keszthely', '1995-01', '2024-12'),
    ('Keszthely', '1961-01', '2024-12'),
    ('Oradea', '1961-01', '1990-12'),
    ('Oradea', '1995-01', '2024-12'),
    ('Oradea', '1961-01', '2024-12'),
    ('Pecs', '1961-01', '1990-12'),
    ('Pecs', '1995-01', '2024-12'),
    ('Pecs', '1961-01', '2024-12')
]

# Futtatás
create_all_global_networks_pca(specifications)

--- Munka kezdete: Cluj : 1961-01 - 1990-12 ---
Felhasznált változók a PCA-hoz: ['mean_tn', 'mean_tx', 'mean_tg', 'rr_sum', 'mean_qq', 'mean_hu']
Magyarázott variancia aránya a komponensekkel: [0.77928702 0.14601583]
Kezdeti hálózat - Csomópontok: 360, Élek: 64619
Pruning 48599 edges to reach average degree of 89...
Finished. Final Average Degree: 89.00
Küszöbérték (0.8120) mentve: Cluj_1961-01_1990-12
PCA súlyok mentve: Cluj_1961-01_1990-12
Hálózat mentve: ../global_networks/rich_global/Cluj_1961-01_1990-12.graphml
----------------------------------------
--- Munka kezdete: Cluj : 1995-01 - 2024-12 ---
Felhasznált változók a PCA-hoz: ['mean_tn', 'mean_tx', 'mean_tg', 'rr_sum', 'mean_qq', 'mean_hu']
Magyarázott variancia aránya a komponensekkel: [0.76773059 0.15711712]
Kezdeti hálózat - Csomópontok: 360, Élek: 64619
Pruning 48599 edges to reach average degree of 89...
Finished. Final Average Degree: 89.00
Küszöbérték (0.8200) mentve: Cluj_1995-01_2024-12
PCA súlyok mentve: Cluj_1995-01